# Decade Classifier Inference Demo

This notebook demonstrates how to use the trained decade classifier model for inference.

In [ ]:
# Setup
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Add project root to path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from scripts.inference import ModelInference

## 1. Load Model

In [ ]:
# Find available checkpoints
experiments_dir = project_root / "experiments"
checkpoints = list(experiments_dir.glob("*/checkpoints/best_checkpoint.pth"))

print("Available checkpoints:")
for i, cp in enumerate(checkpoints):
    print(f"{i}: {cp.parent.parent.name} - {cp}")

In [ ]:
# Load model (change index or path as needed)
checkpoint_path = checkpoints[0] if checkpoints else "path/to/your/checkpoint.pth"
multi_task = False  # Set to True for multi-task models

model = ModelInference(
    checkpoint_path=str(checkpoint_path),
    multi_task=multi_task
)

print(f"Model loaded: {model.config.get('model_name', 'unknown')}")
print(f"Multi-task: {model.multi_task}")

## 2. Single Image Prediction

In [ ]:
# Predict on a single image
image_path = "path/to/your/image.jpg"  # Change this to your image

# For demo, let's create a sample image
if not Path(image_path).exists():
    # Create a dummy image for demo
    dummy_img = Image.new('RGB', (260, 260), color='red')
    image_path = "demo_image.jpg"
    dummy_img.save(image_path)

# Make prediction
result = model.predict_image(image_path)

# Display image and results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Show image
img = Image.open(image_path)
ax1.imshow(img)
ax1.axis('off')
ax1.set_title('Input Image')

# Show predictions
if model.multi_task:
    # Multi-task: show both decade and cluster
    decade_probs = [p['confidence'] for p in result['decade']['top3']]
    decade_labels = [p['class'] for p in result['decade']['top3']]
    
    ax2.bar(decade_labels, decade_probs)
    ax2.set_ylim(0, 1)
    ax2.set_xlabel('Decade')
    ax2.set_ylabel('Confidence')
    ax2.set_title(f"Prediction: {result['decade']['prediction']} ({result['decade']['confidence']:.1%})")
else:
    # Single task: show decade only
    all_probs = list(result['decade']['all_probabilities'].values())
    all_labels = list(result['decade']['all_probabilities'].keys())
    
    ax2.bar(all_labels, all_probs)
    ax2.set_ylim(0, 1)
    ax2.set_xlabel('Decade')
    ax2.set_ylabel('Confidence')
    ax2.set_title(f"Prediction: {result['decade']['prediction']} ({result['decade']['confidence']:.1%})")

plt.tight_layout()
plt.show()

# Print detailed results
print("\nDetailed Results:")
print(f"Decade: {result['decade']['prediction']} (confidence: {result['decade']['confidence']:.1%})")
print("\nTop 3 predictions:")
for i, pred in enumerate(result['decade']['top3'], 1):
    print(f"  {i}. {pred['class']}: {pred['confidence']:.1%}")

if model.multi_task and 'cluster' in result:
    print(f"\nCluster: {result['cluster']['prediction']} (confidence: {result['cluster']['confidence']:.1%})")

## 3. Batch Prediction

In [ ]:
# Predict on multiple images
image_dir = Path("path/to/image/directory")  # Change to your directory

# For demo, create some dummy images
if not image_dir.exists():
    image_dir = Path("demo_images")
    image_dir.mkdir(exist_ok=True)
    
    colors = ['red', 'green', 'blue', 'yellow', 'purple']
    for i, color in enumerate(colors):
        img = Image.new('RGB', (260, 260), color=color)
        img.save(image_dir / f"demo_{i}.jpg")

# Get all images
image_paths = list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.png"))
print(f"Found {len(image_paths)} images")

# Batch predict
results = model.predict_batch(image_paths[:5])  # Limit to 5 for demo

# Display results in a grid
n_images = len(results)
fig, axes = plt.subplots(1, n_images, figsize=(4*n_images, 4))
if n_images == 1:
    axes = [axes]

for ax, res in zip(axes, results):
    if res['status'] == 'success':
        img = Image.open(res['image_path'])
        ax.imshow(img)
        ax.set_title(f"{res['decade']['prediction']}\n({res['decade']['confidence']:.0%})")
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, 'Error', ha='center', va='center')
        ax.set_title(Path(res['image_path']).name)
        ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Confidence Analysis

In [ ]:
# Analyze confidence distribution
if len(results) > 0:
    confidences = [r['decade']['confidence'] for r in results if r['status'] == 'success']
    predictions = [r['decade']['prediction'] for r in results if r['status'] == 'success']
    
    # Plot confidence distribution
    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.hist(confidences, bins=20, edgecolor='black')
    plt.xlabel('Confidence')
    plt.ylabel('Count')
    plt.title('Confidence Distribution')
    
    plt.subplot(1, 2, 2)
    from collections import Counter
    pred_counts = Counter(predictions)
    plt.bar(pred_counts.keys(), pred_counts.values())
    plt.xlabel('Decade')
    plt.ylabel('Count')
    plt.title('Prediction Distribution')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Average confidence: {np.mean(confidences):.1%}")
    print(f"Min confidence: {np.min(confidences):.1%}")
    print(f"Max confidence: {np.max(confidences):.1%}")

## 5. Export Results

In [ ]:
# Export results to various formats
import json
import pandas as pd

# Convert to DataFrame
if len(results) > 0:
    df_data = []
    for r in results:
        if r['status'] == 'success':
            row = {
                'image': Path(r['image_path']).name,
                'decade_prediction': r['decade']['prediction'],
                'decade_confidence': r['decade']['confidence'],
            }
            
            # Add top 3 predictions
            for i, pred in enumerate(r['decade']['top3'], 1):
                row[f'top{i}_decade'] = pred['class']
                row[f'top{i}_confidence'] = pred['confidence']
            
            if model.multi_task and 'cluster' in r:
                row['cluster_prediction'] = r['cluster']['prediction']
                row['cluster_confidence'] = r['cluster']['confidence']
            
            df_data.append(row)
    
    df = pd.DataFrame(df_data)
    
    # Display DataFrame
    display(df)
    
    # Save to CSV
    df.to_csv('predictions.csv', index=False)
    print("\nResults saved to predictions.csv")
    
    # Save detailed JSON
    with open('predictions.json', 'w') as f:
        json.dump(results, f, indent=2)
    print("Detailed results saved to predictions.json")

## 6. Interactive Prediction Widget

In [ ]:
# Create an interactive widget for predictions (requires ipywidgets)
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    
    def predict_uploaded(change):
        clear_output()
        
        # Get uploaded file
        uploaded = upload_widget.value
        if not uploaded:
            return
        
        # Save and predict
        for filename, file_info in uploaded.items():
            # Save image
            temp_path = f"temp_{filename}"
            with open(temp_path, 'wb') as f:
                f.write(file_info['content'])
            
            # Predict
            result = model.predict_image(temp_path)
            
            # Display
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
            
            img = Image.open(temp_path)
            ax1.imshow(img)
            ax1.axis('off')
            ax1.set_title(filename)
            
            # Plot predictions
            probs = [p['confidence'] for p in result['decade']['top3']]
            labels = [p['class'] for p in result['decade']['top3']]
            
            ax2.barh(labels, probs)
            ax2.set_xlim(0, 1)
            ax2.set_xlabel('Confidence')
            ax2.set_title(f"Prediction: {result['decade']['prediction']}")
            
            plt.tight_layout()
            plt.show()
            
            # Clean up
            Path(temp_path).unlink()
    
    # Create upload widget
    upload_widget = widgets.FileUpload(
        accept='image/*',
        multiple=False,
        description='Upload Image'
    )
    
    upload_widget.observe(predict_uploaded, names='value')
    
    print("Upload an image to get predictions:")
    display(upload_widget)
    
except ImportError:
    print("ipywidgets not installed. Run: pip install ipywidgets")
    print("For JupyterLab, also run: jupyter labextension install @jupyter-widgets/jupyterlab-manager")